# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [56]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display

import gradio as gr
import sys
import json
from dotenv import load_dotenv
from openai import OpenAI

In [65]:
load_dotenv(override=True)
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

# Separate client just for Whisper transcription
whisper_client = OpenAI(api_key=OPENAI_API_KEY)

In [66]:
def transcribe_audio(audio_path):
    if audio_path is None:
        return ""
    with open(audio_path, "rb") as f:
        transcript = whisper_client.audio.transcriptions.create(
            model="whisper-1",
            file=f,
            language="en"
        )
    return transcript.text

In [67]:
def text_to_speech(text):
    response = whisper_client.audio.speech.create(
        model="tts-1",
        voice="alloy",  # options: alloy, echo, fable, onyx, nova, shimmer
        input=text
    )
    audio_path = "/tmp/response.mp3"
    response.stream_to_file(audio_path)
    return audio_path

In [59]:
def define_term(term):
    glossary = {
        "transformer": "A neural network architecture based on self-attention.",
        "vector embedding": "A dense numerical representation of data in high-dimensional space.",
        "rag": "Retrieval Augmented Generation — combining document retrieval with language generation.",
        "attention": "A mechanism that allows models to focus on relevant parts of input data."
    }
    return glossary.get(term.lower(), f"No definition found for '{term}'.")

In [60]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "define_term",
            "description": "Define a technical term concisely.",
            "parameters": {
                "type": "object",
                "properties": {
                    "term": {
                        "type": "string",
                        "description": "The technical term to define"
                    }
                },
                "required": ["term"]
            }
        }
    }
]

In [61]:

def chat_stream(history, model):

    system_prompt = """
    You are a senior AI engineer and university-level instructor.
    If a user asks for a definition of a specific term,
    use the define_term tool.
    Otherwise, explain clearly with structure and examples.
    """

    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(history)

    # ---- FIRST CALL (non-streamed) ----
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        temperature=0.3
    )

    message = response.choices[0].message

    # ---- TOOL CALL ----
    if message.tool_calls:

        tool_call = message.tool_calls[0]

        args = tool_call.function.arguments
        if isinstance(args, str):
            arguments = json.loads(args)
        else:
            arguments = args

        tool_result = define_term(arguments["term"])

        # Add assistant tool call message
        messages.append({
            "role": "assistant",
            "content": message.content,
            "tool_calls": message.tool_calls
        })

        # Add tool response
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": tool_result
        })

        # SECOND CALL (streamed)
        stream = client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True,
            temperature=0.3
        )

    else:
        stream = client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True,
            temperature=0.3
        )

    full_response = ""

    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            full_response += chunk.choices[0].delta.content
            yield full_response

In [63]:


models = [
    "openai/gpt-4o-mini",
    "anthropic/claude-3-haiku",
    "meta-llama/llama-3.2-3b-instruct"
]

with gr.Blocks() as demo:
    gr.Markdown("# 🧠 Technical AI Tutor (Week 2 Prototype)")

    model_selector = gr.Dropdown(
        models, value=models[0], label="Select Model"
    )

    chatbot = gr.Chatbot(type='messages')
    msg = gr.Textbox(label="Ask a technical question")

    # --- Audio input ---
    with gr.Row():
        audio_input = gr.Audio(
            sources=["microphone"],
            type="filepath",          # gives you a temp file path
            label="Or speak your question"
        )
        transcribe_btn = gr.Button("🎙️ Transcribe & Send")

    def transcribe_and_send(audio_path, chat_history, model):
        text = transcribe_audio(audio_path)
        if not text:
            yield "", chat_history
            return
        # reuse your existing respond generator
        yield from respond(text, chat_history, model)

    def respond(message, chat_history, model):
        chat_history = chat_history or []
        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": ""})
        assistant_index = len(chat_history) - 1

        for partial in chat_stream(chat_history, model):
            chat_history[assistant_index]["content"] = partial
            yield "", chat_history

    # text path
    msg.submit(respond, [msg, chatbot, model_selector], [msg, chatbot])

    # audio path
    transcribe_btn.click(
        transcribe_and_send,
        [audio_input, chatbot, model_selector],
        [msg, chatbot]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.
